## Retrieval-Augmented Generation (RAG)

Large Language Models (LLMs) generate responses primarily from knowledge acquired during training. Consequently, they do not automatically have access to specialised or private collections, such as a locally stored research corpus. **Retrieval-Augmented Generation (RAG)** addresses this limitation by combining an LLM with an external information retrieval system.[^1]

Instead of relying exclusively on the model's internal knowledge, a RAG system first **retrieves information relevant to the user's query** and then provides this information to the LLM as additional context. This is particularly useful when working with specialised corpora that were not part of the model's training data, or collections that are too large to fit into the model's context window.[^1]

### A simplified RAG pipeline can be represented as:

> **User question → retrieve relevant documents → add documents to the context → LLM → generated answer**

RAG does not normally retrain the language model on the external collection. Instead, the external data are made available to the model **at inference time**.[^1]




In [1]:
%pip install --upgrade --force-reinstall \
    "pydantic>=2.12,<2.13" \
    langchain \
    langchain-openai \
    langchain-chroma \
    langchain-docling \
    langchain-community \
    langchain-text-splitters \
    chromadb \
    docling \
    beautifulsoup4 \
    "numpy<2" \
    "pandas>=2.2,<3"

  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached langchain-1.4.0-py3-none-any.whl.metadata (6.2 kB)
  Using cached langchain_openai-1.6.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached langchain_chroma-1.1.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached langchain_docling-3.0.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached langchain_text_splitters-1.1.2-py3-none-any.whl.metadata (3.3 kB)
  Using cached chromadb-1.5.9-cp39-abi3-macosx_11_0_arm64.whl.metadata (5.0 kB)
  Using cached docling-2.127.0-py3-none-any.whl.metadata (11 kB)
  Using cached beautifulsoup4-4.15.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached numpy-1.26.4-cp312-cp312-macosx_11_0_arm64.whl.metadata (61 kB)
  Using cached pandas-2.3.3-cp312-cp312-macosx_11_0_arm64.whl.metadata (91 kB)
  Using cached annotated_types-0.8.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.41.5-cp312-cp31

### Installations (skip if not neccessary)

In [3]:
import sys
!{sys.executable} -m pip install -U langchain-community

In [1]:
import importlib.metadata as md

for package in [
    "docling",
    "langchain-docling",
    "pydantic",
    "langchain",
    "numpy",
]:
    print(package, md.version(package))

from langchain_docling import DoclingLoader

print("Docling import successful")

docling 2.127.0
langchain-docling 3.0.0
pydantic 2.8.2
langchain 1.4.0
numpy 1.26.4


/opt/anaconda3/lib/python3.12/site-packages/pydantic/_internal/_fields.py:161: UserWarning: Field "model_impl" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/pydantic/_internal/_fields.py:161: UserWarning: Field "model_spec" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/pydantic/_internal/_fields.py:161: UserWarning: Field "model_name" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/pydantic/_internal/_fields.py:161: UserWarning: Field "model_version" has conflict with protected namespace "model_".

You may be able to

AttributeError: force_full_page_ocr

In [2]:
import sys
import importlib.metadata as md

print("Python:", sys.executable)

for package in ["docling", "langchain-docling", "pydantic", "pydantic-core"]:
    try:
        print(f"{package}: {md.version(package)}")
    except md.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

Python: /opt/anaconda3/bin/python
docling: 2.127.0
langchain-docling: 3.0.0
pydantic: 2.12.5
pydantic-core: 2.41.5


In [4]:
import sys
import numpy as np

print(sys.executable)
print(np.__version__)
print(np.__file__)

/opt/anaconda3/bin/python
1.26.4
/opt/anaconda3/lib/python3.12/site-packages/numpy/__init__.py


In [3]:
import sys

!{sys.executable} -m pip install \
    --upgrade \
    --force-reinstall \
    --no-cache-dir \
    "pydantic==2.13.5"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 10.9 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.16.0
    Uninstalling typing_extensions-4.16.0:
      Successfully uninstalled typing_extensions-4.16.0
  Attempting uninstall: annotated-types
    Found existing installation: annotated-types 0.6.0
    Uninstalling annotated-types-0.6.0:
      Successfully uninstalled annotated-types-0.6.0
  Attempting uninstall: typing-inspection
    Found existing installation: typing-inspection 0.4.4
    Uninstalling typing-inspection-0.4.4:
      Successfully uninstalled typing-inspection-0.4.4
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.20.1
    Uninstalling pydantic_core-2.20.1:
      Successfully uninstalled pydantic_core-2.20.1
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.8.2
    Uninstalling pydantic-2.8.2:
      Successfully uninstalled pydantic-2.8

In [1]:
import sys
import pydantic
import importlib.metadata as md

print("Python:", sys.executable)
print("Pydantic:", pydantic.__version__)
print("Pydantic location:", pydantic.__file__)
print("Docling:", md.version("docling"))
print("LangChain Docling:", md.version("langchain-docling"))

from langchain_docling import DoclingLoader

print("Docling import successful")

Python: /opt/anaconda3/bin/python
Pydantic: 2.13.5
Pydantic location: /opt/anaconda3/lib/python3.12/site-packages/pydantic/__init__.py
Docling: 2.127.0
LangChain Docling: 3.0.0
Docling import successful


In [2]:
import sys
!{sys.executable} -m pip check

thinc 8.3.6 has requirement numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4.
opencv-python 5.0.0.93 has requirement numpy>=2; python_version >= "3.9", but you have numpy 1.26.4.
streamlit 1.37.1 has requirement protobuf<6,>=3.20, but you have protobuf 7.35.1.


### Main imports

In [5]:
import os
import warnings
import logging

import bs4

from langchain.agents import AgentState, create_agent
from langchain.messages import MessageLikeRepresentation
from langchain.tools import tool

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_docling import DoclingLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

### Environment setup

For this step you will need to: 
- get a Langchain API Key (https://docs.langchain.com/oss/python/deepagents/rag)
- be added DHInfra project by Florian and get DHInfa API kez

In [6]:
from openai import OpenAI

In [7]:
os.environ["LANGCHAIN_API_KEY"] = "" # insert your own Langchain key
os.environ["LANGCHAIN_TRACING_V2"] = "false"  # <-- FIX 1: Disabled to prevent 403 error
os.environ["LANGCHAIN_PROJECT"] = "DHInfra-Tracing-Demo"
os.environ["LANGSMITH_DISABLE_RUN_COMPRESSION"] = "true"
os.environ["USER_AGENT"] = "my_agent"
os.environ["DHINFRA_API_KEY"] = "" # insert DHInfra key

In [8]:
# <-- FIX 2: Custom class to prevent the 422 "null content" error
class SanitizedChatOpenAI(ChatOpenAI):
    def _get_request_payload(self, input_, *args, **kwargs):
        payload = super()._get_request_payload(input_, *args, **kwargs)
        if "messages" in payload:
            for msg in payload["messages"]:
                if msg.get("content") is None:
                    msg["content"] = ""
        return payload

# Initialize chat model using the sanitized class
model = SanitizedChatOpenAI(
    model="qwen3.5-397b",
    openai_api_key=os.environ["DHINFRA_API_KEY"],
    openai_api_base="https://api.dhinfra.uni-graz.at/v1",
    model_kwargs={"parallel_tool_calls": False}
)



# Initialize embedding model
embeddings = OpenAIEmbeddings(
    model="qwen3-embedding-8b",
    openai_api_key=os.environ["DHINFRA_API_KEY"],
    openai_api_base="https://api.dhinfra.uni-graz.at/v1"
)

vector_store = Chroma(
    collection_name="migraanno_newspapers_v2",
    embedding_function=embeddings,
    persist_directory="./chroma_migraanno",
)


print("Chat model (Qwen), embedding model (Qwen3-Embedding-8B), and Chroma vector store setup done")

Chat model (Qwen), embedding model (Qwen3-Embedding-8B), and Chroma vector store setup done


## Adapting the code for our own data

If it is a dataframe:

In [10]:
import warnings
import logging
import os
import pandas as pd

warnings.filterwarnings("ignore")
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores.utils import filter_complex_metadata


In [11]:
import pandas as pd
from langchain_core.documents import Document

In [12]:
print("Documents in Chroma:", vector_store._collection.count())

Documents in Chroma: 96871


## Loading the embeddings (so no need for re-indexing)

In [16]:
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent


# 1. Define the Chroma retrieval tool.
@tool
def retrieve_context(
    query: str,
    start_year: int | None = None,
    end_year: int | None = None,
    category: str | None = None,
    sentiment: str | None = None,
    k: int = 4,
) -> str:
    """
    Retrieve relevant passages from the historical newspaper corpus.

    The query may be in English, German, or both. Year, category,
    and sentiment filters are optional.
    """

    # Build optional Chroma metadata filters.
    conditions = []

    if start_year is not None:
        conditions.append({
            "year": {"$gte": int(start_year)}
        })

    if end_year is not None:
        conditions.append({
            "year": {"$lte": int(end_year)}
        })

    if category:
        conditions.append({
            "category": {"$eq": category.strip()}
        })

    if sentiment:
        conditions.append({
            "sentiment": {"$eq": sentiment.strip().lower()}
        })

    # Prepare the final metadata filter.
    if not conditions:
        metadata_filter = None
    elif len(conditions) == 1:
        metadata_filter = conditions[0]
    else:
        metadata_filter = {"$and": conditions}

    # Retrieve documents using semantic similarity and metadata filters.
    retrieved_docs = vector_store.similarity_search(
        query=query,
        k=max(1, min(int(k), 10)),
        filter=metadata_filter,
    )

    if not retrieved_docs:
        return "No relevant newspaper passages were found."

    # Convert the retrieved Document objects into text for the agent.
    serialized_documents = []

    for number, doc in enumerate(retrieved_docs, start=1):
        metadata = doc.metadata

        serialized_documents.append(
            f"""
SOURCE {number}

METADATA
Document ID: {metadata.get("chunk_id", "unknown")}
Newspaper: {metadata.get("newspaper_title", "unknown")}
Date: {metadata.get("date", "unknown")}
Year: {metadata.get("year", "unknown")}
Topic: {metadata.get("topic", "unknown")}
Original label: {metadata.get("name_original", "unknown")}
Category: {metadata.get("category", "unknown")}
Sentiment: {metadata.get("sentiment", "unknown")}
Relevancy probability: {metadata.get("relevancy_proba", "unknown")}

PRECEDING TEXT
{metadata.get("preceding_document") or "[No preceding text]"}

RETRIEVED TEXT
{doc.page_content or "[No retrieved text]"}

FOLLOWING TEXT
{metadata.get("following_document") or "[No following text]"}
""".strip()
        )

    separator = "\n\n" + "=" * 80 + "\n\n"
    return separator.join(serialized_documents)


# 2. Define the research assistant's behaviour.
system_prompt = """
You are a research assistant working with a corpus of historical newspapers.

Always use the retrieve_context tool before answering questions about the
corpus. Base your answer only on the passages returned by the tool.

The questions may be in English, while the newspaper texts are mainly in
German. Create a multilingual English-German retrieval query and include
historical terminology or spelling variants when useful.

Extract metadata filters from the user's question:

- use start_year and end_year for a stated period;
- for one exact year, use that year as both start_year and end_year;
- apply category only when the user specifies one;
- apply sentiment only when the user specifies one;
- do not invent filters that the user did not request.

In your answer:

- summarize the evidence;
- distinguish evidence from interpretation;
- cite document IDs, dates, and newspaper titles;
- identify differences or contradictions between sources;
- state clearly when the retrieved evidence is insufficient.
"""


# 3. Register the tool and create the agent.
tools = [retrieve_context]

agent = create_react_agent(
    model=model,
    tools=tools,
    prompt=system_prompt,
)

print("Historical newspaper research assistant created successfully!")

Historical newspaper research assistant created successfully!


## Search newspapers function

In [17]:
query = (
    "What was the relationship between Croats and the government "
    "between 1860 and 1900?"
)

for step in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What was the relationship between Croats and the government between 1860 and 1900?
================================== Ai Message ==================================
Tool Calls:
  retrieve_context (chatcmpl-tool-9ba91e2c50c1ef9e)
 Call ID: chatcmpl-tool-9ba91e2c50c1ef9e
  Args:
    query: Croats Croatia government relationship Kroaten Kroatien Regierung Verhältnis Österreich-Ungarn
    start_year: 1860
    end_year: 1900
    k: 8
================================= Tool Message =================================
Name: retrieve_context

SOURCE 1

METADATA
Document ID: 167871.0
Newspaper: nfp
Date: 1883-04-01
Year: 1883
Topic: 4
Original label: 4_ungarn_ungarischen_ungarische_ungarns
Category: MIN
Sentiment: neutral
Relevancy probability: 0.9813107

PRECEDING TEXT
Unter Dankesworten an den LanoeS-Ehef,die Theilnehmer der Conferenz und den Kärntner Geschichtsverein wurde die Conferenz um halb 2 Uhr geschlossen.



In [18]:
query = (
    "Negative texts towards migration between 1900 and 1938 in the MIG category? "
    
)

for step in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Negative texts towards migration between 1900 and 1938 in the MIG category? 
================================== Ai Message ==================================
Tool Calls:
  retrieve_context (chatcmpl-tool-bc1e7a45b98b60f0)
 Call ID: chatcmpl-tool-bc1e7a45b98b60f0
  Args:
    query: migration negative Einwanderung Auswanderung Fremde Ausländer
    start_year: 1900
    end_year: 1938
    category: MIG
    sentiment: negative
    k: 4
================================= Tool Message =================================
Name: retrieve_context

SOURCE 1

METADATA
Document ID: 330036.0
Newspaper: aze
Date: 1911-04-24
Year: 1911
Topic: 1710
Original label: 1710_ausländer_auswanderung_ausweisung_ausländern
Category: MIG
Sentiment: negative
Relevancy probability: 0.99997044

PRECEDING TEXT
Die Bestimmung öffnet der Polizeiwillkür Tür und Tor. Sie bildet ein Stück aus der Kalaner Jurisprudenz, nach der diejenigen Persone

In [19]:
query = (
    "How were Serbs discussed after 1914? "
    
)

for step in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

How were Serbs discussed after 1914? 
================================== Ai Message ==================================
Tool Calls:
  retrieve_context (chatcmpl-tool-9c2e9bede7175af9)
 Call ID: chatcmpl-tool-9c2e9bede7175af9
  Args:
    query: Serbs Serben Serbia Serbien
    start_year: 1914
    k: 4
================================= Tool Message =================================
Name: retrieve_context

SOURCE 1

METADATA
Document ID: 230127.0
Newspaper: dvb
Date: 1916-07-18
Year: 1916
Topic: 17
Original label: 17_serbien_serbischen_serbische_serben
Category: MIN
Sentiment: neutral
Relevancy probability: 0.99972194

PRECEDING TEXT
Auf den Feldern der Großgrundbesitzer arbeiten Frauen, Kinder und ältere Männer. Die Ernteaussichten in ganz Galizien sind sehr gute.

RETRIEVED TEXT
Topic: 17
Original label: 17_serbien_serbischen_serbische_serben
Category: MIN
Sentiment: neutral

[Austandsehung der Mühlen in Se

In [20]:
query = (
    "How was migration discussed after 1900? "
    
)

for step in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

How was migration discussed after 1900? 
================================== Ai Message ==================================
Tool Calls:
  retrieve_context (chatcmpl-tool-ae67cb6b2e6af08d)
 Call ID: chatcmpl-tool-ae67cb6b2e6af08d
  Args:
    query: migration Auswanderung Einwanderung Wanderung emigration immigration
    start_year: 1900
    end_year: None
    category: None
    sentiment: None
    k: 4
================================= Tool Message =================================
Name: retrieve_context

Error invoking tool 'retrieve_context' with kwargs {'query': 'migration Auswanderung Einwanderung Wanderung emigration immigration', 'start_year': 1900, 'end_year': 'None', 'category': 'None', 'sentiment': 'None', 'k': 4} with error:
 end_year: Input should be a valid integer, unable to parse string as an integer
 Please fix the error and try again.
================================== Ai Message ============

In [21]:
query = (
    "How did the newspapers discuss Jews in the 20th century?"
    
)

for step in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

How did the newspapers discuss Jews in the 20th century?
================================== Ai Message ==================================
Tool Calls:
  retrieve_context (chatcmpl-tool-a6b74ca003cb0f9d)
 Call ID: chatcmpl-tool-a6b74ca003cb0f9d
  Args:
    query: Jews Jewish Juden jüdisch
    start_year: 1900
    end_year: 1999
    k: 10
================================= Tool Message =================================
Name: retrieve_context

SOURCE 1

METADATA
Document ID: 76681.0
Newspaper: dvb
Date: 1907-11-30
Year: 1907
Topic: 18
Original label: 18_juden_jüdischen_synagoge_jüdische
Category: MIN
Sentiment: neutral
Relevancy probability: 0.99993134

PRECEDING TEXT
Der Bau wird demnächst in Angriff genommen und soll die Bahn bis zum Frühjahre schon dem Verkehre übergeben werden. [Religionsvorträge.] In der St. Antoninspfarr bei Brixen ankommen, um von der Baronin Ernst Schönbürger im Zeichen zu gehen, Dr. G

In [22]:
query = (
    "How did the newspapers discuss Jews in Galiciain the 20th century?"
    
)

for step in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

How did the newspapers discuss Jews in Galiciain the 20th century?
================================== Ai Message ==================================
Tool Calls:
  retrieve_context (chatcmpl-tool-9dd68049421e6801)
 Call ID: chatcmpl-tool-9dd68049421e6801
  Args:
    query: Jews Galicia Juden Galizien
    start_year: 1900
    end_year: 1999
    k: 8
================================= Tool Message =================================
Name: retrieve_context

SOURCE 1

METADATA
Document ID: 137020.0
Newspaper: dvb
Date: 1911-02-21
Year: 1911
Topic: 533
Original label: 533_galizien_galizischen_galizische_galiziens
Category: CTX
Sentiment: neutral
Relevancy probability: 0.999966

PRECEDING TEXT
Aus dem Polenklub. Das Präsidium des Polenklubs trat gestern mittags unter dem Vorsitze des Obmannes Professors Tokorpatki zur Beratung zusammen.

RETRIEVED TEXT
Topic: 533
Original label: 533_galizien_galizischen_galizische_g